# Lab Assignment 8
## Neural Machine Translation for English-to-German using LSTMs
---

### Objective

The goal of this assignment is to build, train, and test a Neural Machine Translation (NMT) model
capable of translating English sentences into German. You will achieve this by adapting a
sequence-to-sequence (Seq2Seq) model based on Long Short-Term Memory (LSTM) networks.

### Provided Materials

A complete working notebook that implements an **English-to-French** translator has been shared with you. Your task is to modify the logic in that
reference code to work for the English-to-German language pair.

**You are expected to write the code yourself.** This notebook tells you *what* each task must
produce — the function names, the variables, and the exact output your code must print — but not
*how* to write it. Adapt the reference notebook.

### Marks

| Task | Description | Marks |
|---|---|---|
| **Task 1** | Data Adaptation | **4** |
| **Task 2** | Model Training | **5** |
| **Task 3** | Inference and Translation | **6** |
| **Task 4** | Beam Search Decoding | **5** |
| | **Total** | **20** |

---

### Before you start

- **Use a GPU.** In Colab: `Runtime → Change runtime type → T4 GPU`. Training 40 epochs takes
  roughly **15–25 minutes on a GPU** and well over an hour on CPU. **Start Task 2 training early**
  and write your Task 4 code while it runs.
- **The reference notebook is incomplete.** It calls a function `decode_sequence(...)` that it
  never defines. Writing that function is part of Task 3.
- Saving to `.h5` prints a "legacy format" warning on current Keras. It is harmless.
- Do **not** change the model architecture. Tasks 1–3 adapt the data and the inference code;
  Task 4 improves the *decoding*, not the model.

---
## Setup

These are the only values given to you. Everything else you write yourself.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense

batch_size  = 64
epochs      = 40          # Task 2 requires at least 40
latent_dim  = 256
num_samples = 10000

print("TensorFlow", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices('GPU')))

TensorFlow 2.20.0
GPU available: True


---
# Task 1: Data Adaptation &nbsp;&nbsp;*(4 marks)*

Your first task is to replace the data pipeline to use the English-to-German dataset.

1. In the data acquisition section of your code, download the English-German data uploaded in classroom.
2. After downloading and unzipping, ensure you update the `data_path` variable to point to the
   correct text file: `deu.txt`.
3. Run the data loading and preprocessing steps. **For your submission, report the following
   statistics** that are printed to the console:
   * `Number of samples:`
   * `Number of unique input tokens:`
   * `Number of unique output tokens:`
   * `Max sequence length for inputs:`

### What your code must create

| Variable | Holds |
|---|---|
| `input_texts`, `target_texts` | the English and German sentences |
| `input_characters`, `target_characters` | the sorted character vocabularies |
| `num_encoder_tokens`, `num_decoder_tokens` | sizes of those vocabularies |
| `max_encoder_seq_length`, `max_decoder_seq_length` | longest input and output |
| `input_token_index`, `target_token_index` | character → index dictionaries |
| `encoder_input_data`, `decoder_input_data`, `decoder_target_data` | the three one-hot tensors |

### Output your code must print

```
Number of samples: ...
Number of unique input tokens: ...
Number of unique output tokens: ...
Max sequence length for inputs: ...
Max sequence length for outputs: ...
```

### Hints

- The German file has the **same three-column tab-separated format** as the French one, so the
  parsing logic does not change: splitting a line on tabs gives `[english, german, attribution]`.
- Wrap every German sentence with a **start marker** (`'\t'`) and an **end marker** (`'\n'`).
  Without them the decoder never learns where a translation begins or ends.
- **Teacher forcing.** `decoder_target_data` is `decoder_input_data` shifted **left by one step**:
  at position `t` the decoder must *predict* the character it was *given* at position `t`. There
  is nothing to predict before the start marker.
- Pad the unused positions of all three tensors with the space character.
- German adds characters English does not have (`ä ö ü ß`, and `Ä Ö Ü`), so expect
  `num_decoder_tokens` to be noticeably larger than `num_encoder_tokens`.

In [ ]:
# ---------------- TASK 1 ----------------

# TODO 1: download and unzip the English-German archive, then set data_path
data_path = None

# TODO 2: read the file and build
#           input_texts, target_texts            (lists of sentences)
#           input_characters, target_characters  (sorted character vocabularies)
#         Wrap every GERMAN sentence with the start marker and the end marker.

# TODO 3: compute
#           num_encoder_tokens, num_decoder_tokens
#           max_encoder_seq_length, max_decoder_seq_length

# TODO 4: build input_token_index and target_token_index  (character -> index)

# TODO 5: print the five statistics listed above

# TODO 6: build the three one-hot tensors
#           encoder_input_data, decoder_input_data, decoder_target_data
#         decoder_target_data is decoder_input_data shifted LEFT by one step.
#         Pad unused positions with the space character.

---
# Task 2: Model Training &nbsp;&nbsp;*(5 marks)*

Next, you will train the LSTM-based Encoder-Decoder model on the new English-German dataset.

1. Modify the code to save your newly trained model with a distinct name (e.g.,
   `s2s_eng_ger.h5`) to differentiate it from the original French model.
2. Train the model on the English-German data for at least **40 epochs**. It is highly
   recommended to use a GPU accelerator in Google Colab for this step.
3. For your submission, provide a **screenshot** of the console output from the final epoch of
   training. The screenshot must clearly show the final values for `loss`, `accuracy`,
   `val_loss`, and `val_accuracy`.

### What your code must create

| Name | Is |
|---|---|
| `encoder_inputs`, `encoder_states` | the encoder input, and its two final states |
| `decoder_inputs` | the decoder input |
| `decoder_lstm`, `decoder_dense` | the decoder **layer objects** — Task 3 reuses these |
| `model` | the full training model, compiled |
| `history` | the return value of `model.fit(...)` |

### Output your code must print

- `model.summary()`
- the final epoch line showing `loss`, `accuracy`, `val_loss`, `val_accuracy`
- confirmation that the model was saved

### Hints

- The encoder keeps only its **final states** — this is the "thought vector". Its outputs are
  discarded.
- The decoder is connected to the encoder by exactly **one argument**. Find it in the reference
  code; without it you have two unrelated LSTMs.
- The decoder LSTM must return a prediction at *every* time step, not just the last one.
- Keep `decoder_lstm` and `decoder_dense` as named variables. In Task 3 you must reuse **these
  same trained layer objects** — building new ones gives you untrained weights.
- Train with `validation_split=0.2` so you have the `val_loss` and `val_accuracy` the task asks for.

In [ ]:
# ---------------- TASK 2 ----------------

# TODO 7:  build the encoder -- encoder_inputs, and encoder_states (its two final states)

# TODO 8:  build the decoder -- decoder_inputs, decoder_lstm, decoder_dense
#          Connect it to the encoder. Keep the LAYER OBJECTS: Task 3 reuses them.

# TODO 9:  build and compile `model`, then print model.summary()

# TODO 10: train for at least 40 epochs with validation_split=0.2, keeping `history`

# TODO 11: save the model under a German-specific name

# TODO 12: print the final epoch's loss, accuracy, val_loss and val_accuracy

---
# Task 3: Inference and Translation &nbsp;&nbsp;*(6 marks)*

After the model is trained, you will test its ability to translate new English sentences into
German.

1. Ensure the inference part of your code is **correctly set up** to use the new German model and
   its corresponding vocabulary.
2. Using your translation function, translate the following five English sentences:
   * `I am very tired.`
   * `You are right.`
   * `He is a doctor.`
   * `Can you help me?`
   * `Let's go home.`
3. Present the translation results clearly in your submission, showing the original English input
   and the model's German translation for each of the five sentences.

### Why inference needs two new models

During training the decoder was always handed the *correct* previous character (teacher forcing).
At inference there is no correct answer, so the decoder must consume **its own predictions**, one
character at a time. The training model cannot do this, so you re-wire the **already-trained
layers** into two new models. No new training happens.

### What your code must create

| Name | Does |
|---|---|
| `encoder_model` | English sentence → the two encoder states. Run **once** per sentence |
| `decoder_model` | (one character + state) → (next-char probabilities + new state). Run **in a loop** |
| `reverse_target_char_index` | index → character, for reading predictions back |
| `encode_input(input_text)` | English string → one-hot array for the encoder |
| `decode_sequence(input_seq)` | one-hot array → German string, by **greedy** decoding |

### Output your code must print

```
ENGLISH             GERMAN (greedy)
---------------------------------------------
I am very tired.    ...
You are right.      ...
He is a doctor.     ...
Can you help me?    ...
Let's go home.      ...
```

### Hints

- `decode_sequence` is the function the reference notebook **calls but never defines**. You are
  writing it. Its loop: encode once, start from the `'\t'` marker, then repeatedly ask
  `decoder_model` for the next character, take the most likely one, stop at `'\n'`.
- **Carry the returned `[h, c]` forward** on every pass of the loop. If you do not, the decoder
  restarts from scratch at each character.
- Feed the character you just produced back in as the next decoder input.
- Reuse `decoder_lstm` and `decoder_dense` from Task 2. Building new layers runs without error and
  produces confident nonsense — a silent failure.

> ### Watch out — step 1 hides a real bug
> With `num_samples = 10000`, `max_encoder_seq_length` comes out as **15**. But
> `"I am very tired."` and `"Can you help me?"` are **16 characters long**. The reference code
> sizes its input array to exactly `max_encoder_seq_length` and then writes past the end of it,
> raising `IndexError`.
>
> Nothing is wrong with the model: the encoder is declared with `shape=(None, ...)`, and that
> `None` means an LSTM will read a sequence of **any** length. Only the array is too small.
> Fixing this is part of "ensure the inference part of your code is correctly set up".

In [ ]:
# ---------------- TASK 3 ----------------

# TODO 13: build encoder_model   (English input -> the two encoder states)

# TODO 14: build decoder_model   ([one char] + [h, c] -> [probabilities] + [new h, new c])
#          Reuse decoder_lstm and decoder_dense from Task 2 -- the SAME layer objects.

# TODO 15: build reverse_target_char_index   (index -> character)


def encode_input(input_text):
    """
    One-hot encode an English string for the encoder.

    Output: array of shape (1, width, num_encoder_tokens)
    """
    # TODO 16: allocate the array. Careful: it must be wide enough for input_text,
    #          which is NOT always max_encoder_seq_length -- see the box above.
    # TODO 17: set a 1. for each character (skip characters not in the vocabulary),
    #          then pad the remaining positions with the space character.
    pass


def decode_sequence(input_seq):
    """
    Greedy decoding: encoded English sentence -> German string.

    This is the function the reference notebook calls but never defines.
    """
    # TODO 18: run encoder_model ONCE to get the initial states
    # TODO 19: build the first decoder input from the start marker
    # TODO 20: loop -- ask decoder_model for the next-character probabilities,
    #          take the most likely character, stop at the end marker (or if the
    #          sentence gets too long), otherwise append it, feed it back in as the
    #          next input, and CARRY [h, c] FORWARD as the new states
    # TODO 21: return the decoded string
    pass


# TODO 22: print the five test sentences and their translations

---
# Task 4: Beam Search Decoding &nbsp;&nbsp;*(5 marks)*

### The problem with greedy decoding

`decode_sequence` is **greedy**: at every step it takes the single most likely next character and
never reconsiders. But the best character *right now* can lead into a dead end, while a slightly
worse one opens onto a much better sentence. Once greedy commits, it cannot back out — which is
why it produces German-shaped non-words.

### Beam search

Beam search keeps the **`k` best partial translations alive** at every step instead of one. At
each step it expands every live beam by its top `k` characters, scores all the candidates, and
keeps only the best `k` overall. A beam that produces `'\n'` is **complete** and is set aside.

A sequence is scored by the **sum of the log probabilities** of its characters:

$$\text{score}(y_1 \dots y_T) = \sum_{t=1}^{T} \log P(y_t \mid y_{<t}, x)$$

Two details you must get right:

1. **Add logs — never multiply probabilities.** Multiplying dozens of numbers below 1 underflows
   to zero. This is the same underflow problem you met with n-gram models.
2. **Normalise by length.** Every extra character adds another negative number, so a raw sum
   always favours the *shortest* translation. Divide by a power of the length:

$$\text{final score} = \frac{\sum_t \log P(y_t)}{|y|^{\alpha}} \qquad \alpha \approx 0.7$$

### What your code must create

| Name | Signature |
|---|---|
| `beam_search_decode` | `(input_seq, k=3, alpha=0.7)` → `(best_translation, best_score)` |

### Output your code must print

**(a)** the `k=1` sanity check — with a beam width of one, beam search must reproduce greedy
**exactly**:

```
greedy    : ...
beam k=1  : ...     <- must match greedy
```

**(b)** the comparison over all five sentences:

```
ENGLISH             GREEDY                    BEAM (k=3)                changed?
--------------------------------------------------------------------------------
...
Beam search changed the output for N of 5 sentences.
```

### Hints

- `decoder_model` already returns the **full probability distribution** over every character.
  Greedy throws all but the top one away — you keep the top `k`.
- `np.argsort(probs)[-k:]` gives the indices of the `k` largest probabilities.
- Represent each beam as a tuple of `(score, text_so_far, its own next input, its own [h, c])`.
- **Every beam must carry its own state.** Sharing one state across beams is the most common bug —
  it makes all beams collapse to the same output.
- Use `np.log(p + 1e-12)` so you never take `log(0)`.
- Stop once you have `k` completed beams, or after `max_decoder_seq_length` steps.
- Do the `k=1` check first. If that does not match greedy, nothing else will be right.
- **You do not need to retrain anything.** Beam search only changes how you read answers out of
  the model you already trained.

In [ ]:
# ---------------- TASK 4 ----------------

def beam_search_decode(input_seq, k=3, alpha=0.7):
    """
    Beam search decoding.

    Input :  input_seq   one-hot encoded English sentence, from encode_input()
             k           beam width
             alpha       length-normalisation exponent
    Output:  (best_translation_string, best_normalised_score)
    """
    # TODO 23: run encoder_model once to get the initial states
    # TODO 24: start with ONE live beam -- (score 0.0, empty text, the start marker,
    #          the initial states) -- and an empty list of completed beams
    # TODO 25: loop up to max_decoder_seq_length times. For every LIVE beam:
    #            - get the next-character probability distribution
    #            - expand it by its top k characters:
    #                new_score = beam_score + log(prob)   <- ADD logs, never multiply
    #                end marker -> the beam is COMPLETE, set it aside
    #                otherwise  -> a new candidate beam, carrying its OWN [h, c]
    #          then keep only the k highest-scoring candidates as the new live beams
    # TODO 26: length-normalise every completed beam:
    #            final_score = total_log_prob / (len(text) ** alpha)
    # TODO 27: return the completed beam with the highest final score
    pass


# TODO 28: the k=1 sanity check -- beam search must reproduce greedy EXACTLY

# TODO 29: print the greedy vs beam comparison table over the five sentences,
#          and how many of them changed

### Your answers

**1.** For how many of the five sentences did beam search change the output? Pick one that
changed and say whether the beam result is actually a *better German translation*, or merely a
higher-scoring one.

*Your answer:*

**2.** Try `alpha = 0.0` (no length normalisation) and `alpha = 1.0` (full normalisation) on the
same five sentences. What happens to the length of the translations, and why?

*Your answer:*

---
## Submission checklist

- [ ] `Runtime → Restart and run all` completes without errors
- [ ] **Task 1** — the five statistics are printed and visible
- [ ] **Task 2** — trained for at least 40 epochs, model saved under a German-specific name, and a
      **screenshot** of the final epoch showing `loss`, `accuracy`, `val_loss`, `val_accuracy`
- [ ] **Task 3** — all five sentences translated and printed (and no `IndexError`)
- [ ] **Task 4** — `beam_search_decode` implemented, the `k=1` sanity check passes, the comparison
      table is printed, and both questions answered
- [ ] Submit the `.ipynb` with all outputs saved

### Common mistakes

1. Forgetting the `'\t'` / `'\n'` markers around the German — the decoder never learns to start or stop.
2. Getting the teacher-forcing shift backwards, or off by one.
3. Building **new** LSTM/Dense layers for inference instead of reusing the trained ones.
4. Not carrying `[h, c]` forward inside the decoding loop.
5. Sizing the encoder input array to `max_encoder_seq_length` and crashing on the two
   16-character test sentences.
6. In beam search: sharing one state across all beams, or multiplying probabilities instead of
   adding logs.